# calibrate_validate_dpwm notebook

This notebook embeds the full code from `calibrate_validate_dpwm.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

import argparse
import os
import re
import sys
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from dpwm_model import DPWM #from another code, where the model formulation is defined

def parse_range_string(s: str) -> Tuple[float, float]:
    """
    Parse strings like "[0.4-1]" or "[0-0.8]" into (low, high).
    """
    if s is None:
        raise ValueError("range string is None")
    s = str(s).strip()
    if not s:
        raise ValueError("empty range string")
    # Accept variations like "[0-1]" or "[ 0.4 - 1 ]"
    m = re.search(
        r"\[\s*([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s*-\s*([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s*\]",
        s,
    )
    if not m:
        raise ValueError(f"Could not parse range string: {s!r}")
    lo = float(m.group(1))
    hi = float(m.group(2))
    if lo > hi:
        lo, hi = hi, lo
    return lo, hi


def safe_corr(x: np.ndarray, y: np.ndarray) -> float:
    """
    Pearson correlation with guards for constant series.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_std = np.std(x)
    y_std = np.std(y)
    if x_std == 0.0 or y_std == 0.0:
        # If both are effectively constant and equal, treat as perfect correlation.
        if np.allclose(x, x[0]) and np.allclose(y, y[0]) and np.allclose(x[0], y[0]):
            return 1.0
        return 0.0
    return float(np.corrcoef(x, y)[0, 1])


def kge_2009(sim: np.ndarray, obs: np.ndarray, eps: float = 1e-9) -> float:
    """
    Kling-Gupta Efficiency (KGE2012 style):
      KGE = 1 - sqrt( (r-1)^2 + (alpha-1)^2 + (beta-1)^2 )
    where:
      r     = corr(sim, obs)
      alpha = std(sim) / std(obs)
      beta  = mean(sim) / mean(obs)
    """
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)

    r = safe_corr(obs, sim)

    obs_mean = float(np.mean(obs))
    sim_mean = float(np.mean(sim))
    obs_std = float(np.std(obs))
    sim_std = float(np.std(sim))

    beta = sim_mean / (obs_mean + eps)
    alpha = sim_std / (obs_std + eps)

    return float(1.0 - np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2))


def kge_2012(sim: np.ndarray, obs: np.ndarray, eps: float = 1e-9) -> float:
    """
    KGE 2012 variant:
      gamma = CV_sim / CV_obs
      KGE = 1 - sqrt( (r-1)^2 + (gamma-1)^2 + (beta-1)^2 )
    """
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)

    r = safe_corr(obs, sim)

    obs_mean = float(np.mean(obs))
    sim_mean = float(np.mean(sim))
    obs_std = float(np.std(obs))
    sim_std = float(np.std(sim))

    beta = sim_mean / (obs_mean + eps)
    cv_sim = sim_std / (sim_mean + eps)
    cv_obs = obs_std / (obs_mean + eps)
    gamma = cv_sim / (cv_obs + eps)
    return float(1.0 - np.sqrt((r - 1.0) ** 2 + (gamma - 1.0) ** 2 + (beta - 1.0) ** 2))


def nse(sim: np.ndarray, obs: np.ndarray, eps: float = 1e-12) -> float:
    """Nash-Sutcliffe Efficiency."""
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)
    denom = np.sum((obs - np.mean(obs)) ** 2)
    if denom <= eps:
        return -np.inf
    return float(1.0 - np.sum((sim - obs) ** 2) / denom)


def log_nse(sim: np.ndarray, obs: np.ndarray, offset: float = 1e-2) -> float:
    """
    LogNSE on log(Q + offset), robust for low-flow assessment.
    """
    c = float(offset)
    if c <= 0.0:
        c = 1e-2
    sim_pos = np.maximum(np.asarray(sim, dtype=float), 0.0)
    obs_pos = np.maximum(np.asarray(obs, dtype=float), 0.0)
    return nse(np.log(sim_pos + c), np.log(obs_pos + c))


def log_kge_q(sim_q: np.ndarray, obs_q: np.ndarray, offset: float = 1e-2) -> float:
    """
    Logarithmic KGE on discharge: KGE-2009 applied to log(Q + c).

    Uses mean(log Q_s)/mean(log Q_o), std ratio, and Pearson r on the log series
    (common in the literature); independent of --kge_mode.
    """
    c = float(offset)
    if c <= 0.0:
        c = 1e-2
    sim_pos = np.maximum(np.asarray(sim_q, dtype=float), 0.0)
    obs_pos = np.maximum(np.asarray(obs_q, dtype=float), 0.0)
    log_s = np.log(sim_pos + c)
    log_o = np.log(obs_pos + c)
    return float(kge_2009(log_s, log_o))


def lyne_hollick_baseflow(
    q: np.ndarray, alpha: float = 0.925, n_passes: int = 3
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Lyne-Hollick digital filter for quickflow/baseflow separation.
    Returns (qb, qf) where qb is baseflow and qf is quickflow.
    """
    q = np.asarray(q, dtype=float).reshape(-1)
    q = np.maximum(q, 0.0)
    if q.size == 0:
        return q.copy(), q.copy()

    a = float(np.clip(alpha, 0.0, 0.9999))
    n_passes = int(max(1, n_passes))

    def single_pass(q_in: np.ndarray) -> np.ndarray:
        qf = np.zeros_like(q_in, dtype=float)
        for i in range(1, q_in.size):
            qf_i = a * qf[i - 1] + ((1.0 + a) / 2.0) * (q_in[i] - q_in[i - 1])
            # Keep physically meaningful quickflow bounds.
            qf_i = min(max(qf_i, 0.0), q_in[i])
            qf[i] = qf_i
        return qf

    qf = np.zeros_like(q, dtype=float)
    # Commonly used 3-pass sequence: forward, backward, forward.
    for p in range(n_passes):
        if p % 2 == 0:  # forward pass
            qf = single_pass(q - qf + qf * 0.0) if p > 0 else single_pass(q)
        else:  # backward pass
            q_rev = q[::-1]
            qf_rev = single_pass(q_rev)
            qf = qf_rev[::-1]

    qb = np.maximum(q - qf, 0.0)
    return qb, qf


def bfi_from_baseflow(qb: np.ndarray, q: np.ndarray, eps: float = 1e-12) -> float:
    q_sum = float(np.sum(np.maximum(np.asarray(q, dtype=float), 0.0)))
    if q_sum <= eps:
        return np.nan
    qb_sum = float(np.sum(np.maximum(np.asarray(qb, dtype=float), 0.0)))
    return qb_sum / q_sum


def runoff_coefficient_p_over_r(
    p: np.ndarray, r: np.ndarray, eps: float = 1e-12
) -> float:
    """
    Runoff coefficient: RC = sum(R) / sum(P),
    with P and R in mm/day over the same period.
    """
    p_sum = float(np.sum(np.maximum(np.asarray(p, dtype=float), 0.0)))
    r_sum = float(np.sum(np.maximum(np.asarray(r, dtype=float), 0.0)))
    if p_sum <= eps:
        return np.nan
    return r_sum / p_sum


def kge_bfi_component(
    q_sim: np.ndarray,
    q_obs: np.ndarray,
    kge_mode: str,
    lh_alpha: float,
    lh_passes: int,
) -> Tuple[float, float, float]:
    """
    Compute KGE between separated baseflow time series (qb_sim vs qb_obs),
    and return (KGE_BFI, BFI_sim, BFI_obs).
    """
    qb_sim, _ = lyne_hollick_baseflow(q_sim, alpha=lh_alpha, n_passes=lh_passes)
    qb_obs, _ = lyne_hollick_baseflow(q_obs, alpha=lh_alpha, n_passes=lh_passes)
    bfi_sim = bfi_from_baseflow(qb_sim, q_sim)
    bfi_obs = bfi_from_baseflow(qb_obs, q_obs)
    if kge_mode == "kge2009":
        kge_fn = kge_2009
    else:
        kge_fn = kge_2012
    return float(kge_fn(qb_sim, qb_obs)), float(bfi_sim), float(bfi_obs)


def objective_F(
    sim_q: np.ndarray,
    obs_q: np.ndarray,
    sim_et: np.ndarray,
    obs_et: np.ndarray,
    idx: np.ndarray,
    inv_flow_offset: float = 1.5,
    kge_mode: str = "kge2009",
) -> float:
    """
    F = 1/3 * (KGE(Q) + KGE(1/Q) + KGE(ET))
    """
    sim_q = np.asarray(sim_q, dtype=float)
    obs_q = np.asarray(obs_q, dtype=float)
    sim_et = np.asarray(sim_et, dtype=float)
    obs_et = np.asarray(obs_et, dtype=float)

    if idx.size == 0:
        return -np.inf

    q_obs = obs_q[idx]
    q_sim = sim_q[idx]
    et_obs = obs_et[idx]
    et_sim = sim_et[idx]

    # Mask non-finite values (should already be handled, but keep it robust)
    m = np.isfinite(q_obs) & np.isfinite(q_sim) & np.isfinite(et_obs) & np.isfinite(et_sim)
    if not np.any(m):
        return -np.inf
    q_obs = q_obs[m]
    q_sim = q_sim[m]
    et_obs = et_obs[m]
    et_sim = et_sim[m]

    # Inverse-flow KGE is numerically unstable near zero flow if using 1/Q directly.
    # Use the common offset form 1/(Q + c), with c=1.0 by default.
    c = float(inv_flow_offset)
    if c <= 0.0:
        c = 1.0
    q_obs_pos = np.maximum(q_obs, 0.0)
    q_sim_pos = np.maximum(q_sim, 0.0)
    inv_obs = 1.0 / (q_obs_pos + c)
    inv_sim = 1.0 / (q_sim_pos + c)

    if kge_mode == "kge2009":
        kge_fn = kge_2009
    elif kge_mode == "kge2012":
        kge_fn = kge_2012
    else:
        raise ValueError(f"Unsupported kge_mode={kge_mode!r}. Use 'kge2009' or 'kge2012'.")

    return float(
        (kge_fn(q_sim, q_obs) + kge_fn(inv_sim, inv_obs) + kge_fn(et_sim, et_obs)) / 3.0
    )


def objective_components(
    sim_q: np.ndarray,
    obs_q: np.ndarray,
    sim_et: np.ndarray,
    obs_et: np.ndarray,
    idx: np.ndarray,
    inv_flow_offset: float = 1.5,
    kge_mode: str = "kge2009",
    lognse_offset: float = 1e-2,
    lh_alpha: float = 0.925,
    lh_passes: int = 3,
) -> Tuple[float, float, float, float, float, float, float, float, float]:
    """
    Returns component scores:
      (KGE_Q, KGE_invQ, KGE_ET, F_mean, logNSE_Q, KGE_BFI, BFI_sim, BFI_obs, KGE_logQ)
    KGE_logQ is log-KGE: KGE-2009 on log(Q + lognse_offset), same c as logNSE.
    """
    sim_q = np.asarray(sim_q, dtype=float)
    obs_q = np.asarray(obs_q, dtype=float)
    sim_et = np.asarray(sim_et, dtype=float)
    obs_et = np.asarray(obs_et, dtype=float)

    if idx.size == 0:
        return -np.inf, -np.inf, -np.inf, -np.inf, -np.inf, -np.inf, np.nan, np.nan, -np.inf

    q_obs = obs_q[idx]
    q_sim = sim_q[idx]
    et_obs = obs_et[idx]
    et_sim = sim_et[idx]

    m = np.isfinite(q_obs) & np.isfinite(q_sim) & np.isfinite(et_obs) & np.isfinite(et_sim)
    if not np.any(m):
        return -np.inf, -np.inf, -np.inf, -np.inf, -np.inf, -np.inf, np.nan, np.nan, -np.inf
    q_obs = q_obs[m]
    q_sim = q_sim[m]
    et_obs = et_obs[m]
    et_sim = et_sim[m]

    if kge_mode == "kge2009":
        kge_fn = kge_2009
    elif kge_mode == "kge2012":
        kge_fn = kge_2012
    else:
        raise ValueError(f"Unsupported kge_mode={kge_mode!r}. Use 'kge2009' or 'kge2012'.")

    c = float(inv_flow_offset)
    if c <= 0.0:
        c = 1.0
    q_obs_pos = np.maximum(q_obs, 0.0)
    q_sim_pos = np.maximum(q_sim, 0.0)
    inv_obs = 1.0 / (q_obs_pos + c)
    inv_sim = 1.0 / (q_sim_pos + c)

    kge_q = float(kge_fn(q_sim, q_obs))
    kge_invq = float(kge_fn(inv_sim, inv_obs))
    kge_et = float(kge_fn(et_sim, et_obs))
    f_mean = float((kge_q + kge_invq + kge_et) / 3.0)
    kge_logq = float(log_kge_q(q_sim_pos, q_obs_pos, offset=lognse_offset))
    lognse_q = float(log_nse(q_sim_pos, q_obs_pos, offset=lognse_offset))
    kge_bfi, bfi_sim, bfi_obs = kge_bfi_component(
        q_sim_pos, q_obs_pos, kge_mode=kge_mode, lh_alpha=lh_alpha, lh_passes=lh_passes
    )
    return kge_q, kge_invq, kge_et, f_mean, lognse_q, kge_bfi, bfi_sim, bfi_obs, kge_logq


class DBFReader:
    def __init__(self, path: str) -> None:
        import struct

        self.path = path
        with open(path, "rb") as f:
            header = f.read(32)
            self.version = header[0]
            self.num_records = int.from_bytes(header[4:8], byteorder="little", signed=False)
            self.header_len = int.from_bytes(header[8:10], byteorder="little", signed=False)
            self.record_len = int.from_bytes(header[10:12], byteorder="little", signed=False)

            # Read field descriptors (32 bytes each) until terminator 0x0D
            f.seek(32)
            self.fields: List[Tuple[str, str, int, int, int]] = []
            # Each field descriptor includes name (11), type (1), address (4?), length (1), decimal (1)
            # We'll compute offsets inside record payload ourselves.
            offset = 0
            while True:
                chunk = f.read(32)
                if not chunk or chunk[0] == 13:
                    break
                name = chunk[0:11].split(b"\x00", 1)[0].decode("ascii", "replace").strip()
                field_type = chunk[11:12].decode("ascii", "replace").strip()
                length = int(chunk[16])
                dec = int(chunk[17])
                self.fields.append((name, field_type, length, dec, offset))
                offset += length

    def read_records(self, field_names: Sequence[str], where_basin_ids: Optional[Iterable[str]] = None) -> Dict[str, Dict[str, str]]:
        """
        Returns: { basin_id: {field_name: raw_string_value, ...}, ... }
        """
        field_set = set(field_names)
        want_basin_ids = None if where_basin_ids is None else set(where_basin_ids)

        # Find basin_id field index/offset
        basin_field = None
        field_defs = []
        for name, ftype, length, dec, offset in self.fields:
            if name == "basin_id":
                basin_field = (name, ftype, length, dec, offset)
            if name in field_set:
                field_defs.append((name, ftype, length, dec, offset))
        if basin_field is None:
            raise ValueError("DBF missing 'basin_id' field")
        basin_offset = basin_field[4]
        basin_len = basin_field[2]

        out: Dict[str, Dict[str, str]] = {}
        with open(self.path, "rb") as f:
            for i in range(self.num_records):
                f.seek(self.header_len + i * self.record_len)
                del_flag = f.read(1)
                if not del_flag:
                    break
                record_payload = f.read(self.record_len - 1)
                if del_flag != b" ":
                    continue

                basin_raw = record_payload[basin_offset : basin_offset + basin_len].decode("ascii", "replace").strip()
                if not basin_raw:
                    continue
                if want_basin_ids is not None and basin_raw not in want_basin_ids:
                    continue

                vals: Dict[str, str] = {}
                for name, ftype, length, dec, offset in field_defs:
                    raw = record_payload[offset : offset + length].decode("ascii", "replace").rstrip()
                    vals[name] = raw.strip()
                out[basin_raw] = vals
        return out


@dataclass(frozen=True)
class BasinCalibrationConfig:
    warm_span_days: int = 365
    train_frac: float = 0.70
    n_days_total: int = 4018
    seed: int = 42


def lsade_optimize(
    objective_minimize: callable,
    bounds_low: np.ndarray,
    bounds_high: np.ndarray,
    *,
    max_gens: int = 25,
    np_max: int = 40,
    np_min: int = 6,
    memory_size: int = 10,
    p_best_frac: float = 0.2,
    seed: int = 42,
    progress_label: Optional[str] = None,
    progress_every: int = 5,
) -> Tuple[np.ndarray, float]:
    """
    LSHADE-inspired differential evolution with:
      - success-history adaptation for F and CR (SHADE-like)
      - linear population size reduction
      - DE mutation with p-best strategy
    This is an implementation that matches the high-level LSHADE/SHADE behavior,
    but is not a byte-for-byte port of Tanabe & Fukunaga (2014).
    """
    rng = np.random.default_rng(seed)
    d = int(bounds_low.size)
    if d != len(bounds_high):
        raise ValueError("bounds_low/high mismatch")

    # Population initialization
    x = bounds_low + rng.random((np_max, d)) * (bounds_high - bounds_low)
    if progress_label is not None:
        print(f"  {progress_label}: LSADE initial population ({np_max} evaluations)...")
    fvals = np.array([objective_minimize(xi) for xi in x], dtype=float)

    # Archive for diversity
    archive: List[np.ndarray] = []
    archive_max = np_max

    # Success-history memories
    mf = np.full(memory_size, 0.5, dtype=float)
    mcr = np.full(memory_size, 0.5, dtype=float)
    mem_index = 0

    best_idx = int(np.argmin(fvals))
    best_x = x[best_idx].copy()
    best_f = float(fvals[best_idx])

    def current_np(gen: int) -> int:
        # Linear reduction from np_max to np_min across max_gens
        t = gen / max(1, max_gens - 1)
        return int(round(np_max - (np_max - np_min) * t))

    pbest_min_count = max(1, int(np.ceil(p_best_frac * np_max)))

    for gen in range(max_gens):
        np_curr = current_np(gen)
        if np_curr < np_min:
            np_curr = np_min

        # Sort by fitness
        idx_sorted = np.argsort(fvals)
        # Determine best candidates for p-best selection (use current population subset)
        idx_sorted = idx_sorted[:np_curr]
        best_candidates = idx_sorted[: max(1, int(np.ceil(p_best_frac * np_curr)))]

        # Evaluate for each individual in current subset
        successes_f = []
        successes_cr = []
        successes_w = []

        for i in range(np_curr):
            # Pick memory index
            k = int(rng.integers(0, memory_size))
            # Sample F and CR from memories (SHADE-like)
            # F uses Cauchy around mf, CR uses Normal around mcr
            f_i = mf[k]
            cr_i = mcr[k]
            # Resample F until valid
            for _ in range(20):
                f_i = float(rng.standard_cauchy() * 0.1 + mf[k])
                if f_i > 0:
                    break
            # Clip to [0,1] for stability
            f_i = float(np.clip(f_i, 1e-12, 1.0))
            cr_i = float(np.clip(rng.normal(mcr[k], 0.1), 0.0, 1.0))

            # p-best index (among best candidates)
            pbest_i = int(rng.choice(best_candidates))

            # Mutation: x_pbest + F*(x_r1 - x_r2) + (optional) + something
            # We'll use: v = x_i + F*(x_pbest - x_i) + F*(x_r1 - x_r2)
            # r1,r2 picked from population + archive
            pool = np.vstack([x[:np_curr], np.array(archive)]) if len(archive) > 0 else x[:np_curr]
            # Select r1 and r2 distinct and not i
            r_indices = list(range(np_curr))
            r_indices.remove(i)
            r1 = int(rng.choice(r_indices))
            r_indices_2 = [j for j in r_indices if j != r1]
            r2 = int(rng.choice(r_indices_2))

            x_i = x[i]
            x_pbest = x[pbest_i]
            x_r1 = x[r1]
            x_r2 = x[r2]

            v = x_i + f_i * (x_pbest - x_i) + f_i * (x_r1 - x_r2)
            # Bound handling: clip (simple but effective)
            v = np.clip(v, bounds_low, bounds_high)

            # Binomial crossover
            j_rand = int(rng.integers(0, d))
            u = x_i.copy()
            cross_mask = rng.random(d) < cr_i
            cross_mask[j_rand] = True
            u[cross_mask] = v[cross_mask]
            # Clip again
            u = np.clip(u, bounds_low, bounds_high)

            f_trial = float(objective_minimize(u))

            # Selection (minimization)
            if f_trial < fvals[i]:
                improvement = fvals[i] - f_trial
                successes_f.append(f_i)
                successes_cr.append(cr_i)
                successes_w.append(improvement)
                # Archive: store replaced parent
                if len(archive) < archive_max:
                    archive.append(x[i].copy())
                x[i] = u
                fvals[i] = f_trial

                if f_trial < best_f:
                    best_f = f_trial
                    best_x = u.copy()

        # Update memories using successes
        if len(successes_w) > 0 and np.sum(successes_w) > 0:
            w = np.array(successes_w, dtype=float)
            w = w / (np.sum(w) + 1e-12)
            f_succ = np.array(successes_f, dtype=float)
            cr_succ = np.array(successes_cr, dtype=float)
            # Lehmer mean for F (as in SHADE)
            mf[mem_index] = float(np.sum(w * (f_succ**2)) / (np.sum(w * f_succ) + 1e-12))
            # Weighted mean for CR
            mcr[mem_index] = float(np.sum(w * cr_succ))
            mem_index = (mem_index + 1) % memory_size

        if progress_label is not None and (
            (gen + 1) % max(1, progress_every) == 0 or gen == max_gens - 1
        ):
            # best_f is minimized (-KGE); report maximized objective for readability
            print(
                f"  {progress_label}: LSADE gen {gen + 1}/{max_gens} "
                f"best_target={-best_f:.5f} pop={np_curr}"
            )

    return best_x, best_f


def main() -> None:
    if hasattr(sys.stdout, "reconfigure"):
        try:
            sys.stdout.reconfigure(line_buffering=True)
        except (OSError, ValueError):
            pass

    parser = argparse.ArgumentParser(description="DPWM per-basin calibration/validation")
    parser.add_argument(
        "--streamflow_csv",
        default="data/streamflow/estreams_timeseries_streamflow_v02.csv",
    )
    parser.add_argument("--rain_csv", default="Results/forcing/rainplusmelt.csv")
    parser.add_argument("--pet_csv", default="Results/forcing/PET.csv")
    parser.add_argument(
        "--signatures_csv",
        default="data/csv_estream/estreams_hydrometeo_signatures.csv",
    )
    parser.add_argument("--params_dbf", default="data/parameters/Parameters_basins.dbf")
    parser.add_argument(
        "--q_obs_unit",
        choices=["m3s", "mmday"],
        default="m3s",
        help="Unit of observed streamflow in streamflow_csv.",
    )
    parser.add_argument(
        "--area_field",
        default="area_estre",
        help="DBF field containing basin area in km^2 (used when q_obs_unit='m3s').",
    )
    parser.add_argument(
        "--basins",
        default="",
        help="Comma-separated basin IDs. Empty means use basin columns from --rain_csv.",
    )
    parser.add_argument("--n_days_total", type=int, default=4018, help="Total length of the optimization time window (days).")
    parser.add_argument("--warm_span_days", type=int, default=365, help="DPWM warm-up days.")
    parser.add_argument(
        "--max_warm_iterations",
        type=int,
        default=1000,
        help="Max storage-spin-up repeats inside DPWM.simulate (lower = faster, less exact spin-up).",
    )
    parser.add_argument("--train_frac", type=float, default=0.70, help="Calibration fraction among valid days after warm-up.")
    parser.add_argument("--seed", type=int, default=42)
    # Higher defaults to provide a stronger optimization budget.
    parser.add_argument("--max_gens", type=int, default=40)
    parser.add_argument("--np_max", type=int, default=50)
    parser.add_argument("--np_min", type=int, default=10)
    parser.add_argument("--memory_size", type=int, default=10)
    parser.add_argument("--p_best_frac", type=float, default=0.2)
    parser.add_argument(
        "--output_csv",
        default="Results/calibration/calibration_validation_results.csv",
    )
    parser.add_argument("--cache_dir", default="Results/cache_obs")
    parser.add_argument(
        "--inv_flow_offset",
        type=float,
        default=1.5,
        help="Offset c in inverse-flow term 1/(Q+c).",
    )
    parser.add_argument(
        "--kge_mode",
        choices=["kge2009", "kge2012"],
        default="kge2012",
        help="KGE formulation used in objective terms.",
    )
    parser.add_argument(
        "--objective_target",
        choices=[
            "full",
            "invq_only",
            "lognse_only",
            "logkge_only",
            "q_invq_balanced",
            "kge_bfi_only",
            "bfi_q_invq_triple",
            "bfi_q_logkge_triple",
        ],
        default="invq_only",
        help=(
            "Optimization target: full F, invq_only, lognse_only, logkge_only (log-KGE on Q only), "
            "q_invq_balanced, kge_bfi_only, "
            "bfi_q_invq_triple = (KGE_BFI+KGE_invQ+KGE_Q)/3, "
            "bfi_q_logkge_triple = (KGE_BFI+KGE_logQ+KGE_Q)/3 (log-KGE = KGE-2009 on log(Q+lognse_offset))."
        ),
    )
    parser.add_argument(
        "--lognse_offset",
        type=float,
        default=1e-2,
        help="Offset c for logNSE and log-KGE: NSE(log(Q+c)); log-KGE uses KGE-2009 on log(Q+c).",
    )
    parser.add_argument(
        "--lh_alpha",
        type=float,
        default=0.925,
        help="Lyne-Hollick filter alpha for baseflow separation.",
    )
    parser.add_argument(
        "--lh_passes",
        type=int,
        default=3,
        help="Number of Lyne-Hollick passes (e.g., 3 for forward-backward-forward).",
    )
    parser.add_argument(
        "--sm_mode",
        choices=["fixed", "vary"],
        default="vary",
        help="Treat Sm as fixed (DBF value) or optimize in [--sm_abs_min, --sm_abs_max] (default: vary, 0–8000).",
    )
    parser.add_argument(
        "--sm_rel_range",
        type=float,
        default=0.20,
        help="Relative range for Sm when sm_mode='vary' (e.g., 0.20 => +/-20%%).",
    )
    parser.add_argument(
        "--sm_abs_min",
        type=float,
        default=0.0,
        help="Absolute lower bound for Sm when sm_mode='vary' and sm_bounds_mode='absolute'.",
    )
    parser.add_argument(
        "--sm_abs_max",
        type=float,
        default=8000.0,
        help="Absolute upper bound for Sm when sm_mode='vary' and sm_bounds_mode='absolute'.",
    )
    parser.add_argument(
        "--sm_bounds_mode",
        choices=["relative", "absolute"],
        default="absolute",
        help="How Sm bounds are defined when sm_mode='vary'.",
    )
    args = parser.parse_known_args()[0]

    # Load forcing time series from Results/forcing (see prepare_eastream_inputs.py).
    rain_df = pd.read_csv(args.rain_csv)
    pet_df = pd.read_csv(args.pet_csv)
    forcing_dates = rain_df["date"].astype(str).to_numpy()
    # Basin columns are the remaining columns after 'date'
    available_basins = [c for c in rain_df.columns if c != "date"]
    if args.basins.strip():
        basins = [b.strip() for b in args.basins.split(",") if b.strip()]
    else:
        basins = available_basins

    pet_cols = set([c for c in pet_df.columns if c != "date"])
    for b in basins:
        if b not in pet_cols:
            raise ValueError(f"Requested basin {b!r} not found in {args.pet_csv}")

    # Load streamflow observed Q once for all requested basins, aligned to forcing_dates
    os.makedirs(args.cache_dir, exist_ok=True)
    cache_q = os.path.join(args.cache_dir, f"Q_obs_{'_'.join(basins)}.npz")
    if os.path.exists(cache_q):
        data = np.load(cache_q, allow_pickle=True)
        q_obs_aligned = data["q_obs_aligned"]
    else:
        date_to_idx = {d: i for i, d in enumerate(forcing_dates)}
        forcing_set = set(forcing_dates.tolist())

        basin_to_col = {b: i for i, b in enumerate(basins)}
        q_obs_aligned = np.full((len(forcing_dates), len(basins)), np.nan, dtype=float)

        usecols = ["dates"] + basins
        # Chunksize: tradeoff between speed and memory
        chunksize = 100000
        # The streamflow CSV is huge, so we stream it.
        for chunk in pd.read_csv(args.streamflow_csv, usecols=usecols, chunksize=chunksize):
            # Filter rows whose date is in the forcing window
            mask = chunk["dates"].astype(str).isin(forcing_set)
            if not mask.any():
                continue
            sub = chunk.loc[mask]
            # Map dates to indices in the forcing array
            idxs = sub["dates"].astype(str).map(date_to_idx).to_numpy(dtype=int)
            values = sub[basins].to_numpy(dtype=float)
            q_obs_aligned[idxs, :] = values

        np.savez_compressed(cache_q, q_obs_aligned=q_obs_aligned)

    if q_obs_aligned.shape != (len(forcing_dates), len(basins)):
        raise ValueError(
            f"Aligned Q_obs shape {q_obs_aligned.shape} does not match forcing grid "
            f"({len(forcing_dates)} days × {len(basins)} basins). "
            f"If you changed basins or rain_csv, delete cache file {cache_q!r} and re-run."
        )

    for j, b in enumerate(basins):
        n_q_on_forcing = int(np.sum(np.isfinite(q_obs_aligned[:, j])))
        if n_q_on_forcing == 0:
            raise ValueError(
                f"No observed streamflow mapped onto meteorology dates for basin={b!r}. "
                f"Discharge rows in {args.streamflow_csv} must use the same calendar as "
                f"'date' in {args.rain_csv} (e.g. YYYY-MM-DD). Check period overlap and column names."
            )

    # Load signatures for hydro start dates
    sig_df = pd.read_csv(args.signatures_csv)
    sig_df["basin_id"] = sig_df["basin_id"].astype(str)

    params_dbf_path = args.params_dbf
    dbf_reader = DBFReader(params_dbf_path)
    param_field_names = ["basin_id", "alpha 1", "Sm", "alpha 2", "Kf", "Ks", args.area_field]
    param_raw = dbf_reader.read_records(param_field_names, where_basin_ids=basins)

    results = []

    config = BasinCalibrationConfig(
        warm_span_days=args.warm_span_days,
        train_frac=args.train_frac,
        n_days_total=args.n_days_total,
        seed=args.seed,
    )

    # Prepare date -> index mapping for slicing the optimization window by hydro start date.
    forcing_dates_dt = pd.to_datetime(forcing_dates, format="%Y-%m-%d")
    # Ensure sorted
    # (forcing CSV should already be sorted by date)

    for basin_i, basin_id in enumerate(basins):
        if basin_id not in param_raw:
            raise ValueError(f"Parameters_basins DBF has no record for basin_id={basin_id!r}")

        raw = param_raw[basin_id]
        alpha1_lo, alpha1_hi = parse_range_string(raw["alpha 1"])
        alpha2_lo, alpha2_hi = parse_range_string(raw["alpha 2"])
        kf_lo, kf_hi = parse_range_string(raw["Kf"])
        ks_lo, ks_hi = parse_range_string(raw["Ks"])
        sm_fixed = float(raw["Sm"])
        area_km2 = float(raw[args.area_field]) if args.area_field in raw and raw[args.area_field] != "" else np.nan

        # DPWM uses alpha*(2-alpha) and nested square-root expressions.
        # Keep alphas away from exact 0 and exact 1 for numeric stability.
        eps_alpha = 1e-6
        alpha1_lo = max(alpha1_lo, eps_alpha)
        alpha2_lo = max(alpha2_lo, eps_alpha)
        alpha_upper_cap = 0.99
        alpha1_hi = min(alpha1_hi, alpha_upper_cap)
        alpha2_hi = min(alpha2_hi, alpha_upper_cap)
        if alpha1_hi <= alpha1_lo:
            alpha1_hi = min(0.99, alpha1_lo + 1e-4)
        if alpha2_hi <= alpha2_lo:
            alpha2_hi = min(0.99, alpha2_lo + 1e-4)

        # Slicing window: use hydro start date and take n_days_total
        row = sig_df.loc[sig_df["basin_id"] == basin_id]
        if row.empty:
            raise ValueError(f"basin_id={basin_id!r} not found in signatures csv")
        start_hydro = pd.to_datetime(row["start_date_hydro"].iloc[0])
        # Find start index in forcing_dates_dt
        matches = np.where(forcing_dates_dt == start_hydro)[0]
        if matches.size == 0:
            # Fallback: if hydro start is not inside the forcing dates (shouldn't happen for provided sample)
            # choose closest date >= start_hydro.
            start_idx = int(np.searchsorted(forcing_dates_dt.values, start_hydro.to_datetime64(), side="left"))
        else:
            start_idx = int(matches[0])
        end_idx = min(start_idx + config.n_days_total, len(forcing_dates_dt))
        if end_idx - start_idx <= config.warm_span_days + 5:
            raise ValueError(f"Not enough days for basin={basin_id} after slicing to optimization window.")

        precip_full = rain_df[basin_id].to_numpy(dtype=float)
        pet_full = pet_df[basin_id].to_numpy(dtype=float)
        precip = precip_full[start_idx:end_idx]
        pet = pet_full[start_idx:end_idx]

        q_obs = q_obs_aligned[start_idx:end_idx, basin_i]
        # Convert observed Q to model-compatible units when needed.
        # DPWM Q is in depth units (mm/day). If observations are m3/s:
        #   mm/day = Q(m3/s) * 86.4 / area(km2)
        if args.q_obs_unit == "m3s":
            if not np.isfinite(area_km2) or area_km2 <= 0.0:
                raise ValueError(
                    f"Invalid basin area for basin={basin_id}: {area_km2}. "
                    f"Check DBF field '{args.area_field}'."
                )
            q_obs = q_obs * 86.4 / area_km2

        n = len(q_obs)
        warm_span = config.warm_span_days
        valid_mask = np.isfinite(q_obs) & np.isfinite(pet) & (np.arange(n) >= warm_span)
        valid_idx = np.where(valid_mask)[0]
        if valid_idx.size < 50:
            n_q_window = int(np.sum(np.isfinite(q_obs)))
            raise ValueError(
                f"Too few valid calibration/validation days for basin={basin_id}: "
                f"{valid_idx.size} (need >= 50). In this {n}-day window, finite Q days={n_q_window}. "
                "Valid days require finite Q and PET and index >= warm_span. "
                "Meteorology and discharge must overlap in time for this hydro window."
            )

        rng = np.random.default_rng(config.seed + basin_i)
        n_train = int(round(config.train_frac * valid_idx.size))
        n_train = max(1, min(n_train, valid_idx.size - 1))
        train_idx = rng.choice(valid_idx, size=n_train, replace=False)
        val_idx = np.setdiff1d(valid_idx, train_idx, assume_unique=False)

        # Build optimization bounds.
        if args.sm_mode == "vary":
            if args.sm_bounds_mode == "absolute":
                sm_low = max(1e-6, float(args.sm_abs_min))
                sm_high = max(sm_low + 1e-6, float(args.sm_abs_max))
            else:
                sm_low = max(1e-6, sm_fixed * (1.0 - float(args.sm_rel_range)))
                sm_high = max(sm_low + 1e-6, sm_fixed * (1.0 + float(args.sm_rel_range)))
            bounds_low = np.array([alpha1_lo, sm_low, alpha2_lo, kf_lo, ks_lo], dtype=float)
            bounds_high = np.array([alpha1_hi, sm_high, alpha2_hi, kf_hi, ks_hi], dtype=float)
        else:
            bounds_low = np.array([alpha1_lo, alpha2_lo, kf_lo, ks_lo], dtype=float)
            bounds_high = np.array([alpha1_hi, alpha2_hi, kf_hi, ks_hi], dtype=float)

        def vector_to_params(x: np.ndarray) -> List[float]:
            if args.sm_mode == "vary":
                alpha1, sm, alpha2, kf, ks = map(float, x)
                return [alpha1, sm, alpha2, kf, ks]
            alpha1, alpha2, kf, ks = map(float, x)
            return [alpha1, sm_fixed, alpha2, kf, ks]

        def eval_trial(x: np.ndarray) -> float:
            params = vector_to_params(x)
            model = DPWM(
                params,
                warm_span_days=config.warm_span_days,
                max_warm_iterations=args.max_warm_iterations,
            )
            with np.errstate(divide="ignore", invalid="ignore"):
                flux_output, _, _ = model.simulate(precip, pet)
            sim_q = flux_output.Q
            sim_et = flux_output.ET
            if not (np.all(np.isfinite(sim_q)) and np.all(np.isfinite(sim_et))):
                # Invalid simulation trajectory -> very poor objective.
                return float(1e6)
            (
                kge_q,
                kge_invq,
                kge_et,
                f_mean,
                lognse_q,
                kge_bfi,
                bfi_sim,
                bfi_obs,
                kge_logq,
            ) = objective_components(
                sim_q,
                q_obs,
                sim_et,
                pet,
                train_idx,
                inv_flow_offset=args.inv_flow_offset,
                kge_mode=args.kge_mode,
                lognse_offset=args.lognse_offset,
                lh_alpha=args.lh_alpha,
                lh_passes=args.lh_passes,
            )
            if args.objective_target == "invq_only":
                target = kge_invq
            elif args.objective_target == "lognse_only":
                target = lognse_q
            elif args.objective_target == "logkge_only":
                target = kge_logq
            elif args.objective_target == "q_invq_balanced":
                target = 0.5 * (kge_q + kge_invq)
            elif args.objective_target == "kge_bfi_only":
                target = kge_bfi
            elif args.objective_target == "bfi_q_invq_triple":
                target = (kge_bfi + kge_invq + kge_q) / 3.0
            elif args.objective_target == "bfi_q_logkge_triple":
                target = (kge_bfi + kge_logq + kge_q) / 3.0
            else:
                target = f_mean
            # Minimization target (maximize selected target).
            return -float(target)

        def compute_validation_best(x: np.ndarray) -> float:
            params = vector_to_params(x)
            model = DPWM(
                params,
                warm_span_days=config.warm_span_days,
                max_warm_iterations=args.max_warm_iterations,
            )
            with np.errstate(divide="ignore", invalid="ignore"):
                flux_output, _, _ = model.simulate(precip, pet)
            sim_q = flux_output.Q
            sim_et = flux_output.ET
            if not (np.all(np.isfinite(sim_q)) and np.all(np.isfinite(sim_et))):
                return -np.inf
            (
                kge_q,
                kge_invq,
                kge_et,
                f_mean,
                lognse_q,
                kge_bfi,
                bfi_sim,
                bfi_obs,
                kge_logq,
            ) = objective_components(
                sim_q,
                q_obs,
                sim_et,
                pet,
                val_idx,
                inv_flow_offset=args.inv_flow_offset,
                kge_mode=args.kge_mode,
                lognse_offset=args.lognse_offset,
                lh_alpha=args.lh_alpha,
                lh_passes=args.lh_passes,
            )
            if args.objective_target == "invq_only":
                return float(kge_invq)
            if args.objective_target == "lognse_only":
                return float(lognse_q)
            if args.objective_target == "logkge_only":
                return float(kge_logq)
            if args.objective_target == "q_invq_balanced":
                return float(0.5 * (kge_q + kge_invq))
            if args.objective_target == "kge_bfi_only":
                return float(kge_bfi)
            if args.objective_target == "bfi_q_invq_triple":
                return float((kge_bfi + kge_invq + kge_q) / 3.0)
            if args.objective_target == "bfi_q_logkge_triple":
                return float((kge_bfi + kge_logq + kge_q) / 3.0)
            return float(f_mean)

        def compute_components_for_indices(
            x: np.ndarray, idx_subset: np.ndarray
        ) -> Tuple[float, float, float, float, float, float, float, float, float]:
            params = vector_to_params(x)
            model = DPWM(
                params,
                warm_span_days=config.warm_span_days,
                max_warm_iterations=args.max_warm_iterations,
            )
            with np.errstate(divide="ignore", invalid="ignore"):
                flux_output, _, _ = model.simulate(precip, pet)
            sim_q = flux_output.Q
            sim_et = flux_output.ET
            if not (np.all(np.isfinite(sim_q)) and np.all(np.isfinite(sim_et))):
                return (
                    -np.inf,
                    -np.inf,
                    -np.inf,
                    -np.inf,
                    -np.inf,
                    -np.inf,
                    np.nan,
                    np.nan,
                    -np.inf,
                )
            return objective_components(
                sim_q,
                q_obs,
                sim_et,
                pet,
                idx_subset,
                inv_flow_offset=args.inv_flow_offset,
                kge_mode=args.kge_mode,
                lognse_offset=args.lognse_offset,
                lh_alpha=args.lh_alpha,
                lh_passes=args.lh_passes,
            )

        def compute_rc_for_indices(
            x: np.ndarray, idx_subset: np.ndarray
        ) -> Tuple[float, float]:
            params = vector_to_params(x)
            model = DPWM(
                params,
                warm_span_days=config.warm_span_days,
                max_warm_iterations=args.max_warm_iterations,
            )
            with np.errstate(divide="ignore", invalid="ignore"):
                flux_output, _, _ = model.simulate(precip, pet)
            sim_q = flux_output.Q
            if not np.all(np.isfinite(sim_q)):
                return np.nan, np.nan
            q_obs_sub = q_obs[idx_subset]
            q_sim_sub = sim_q[idx_subset]
            p_sub = precip[idx_subset]
            m = np.isfinite(q_obs_sub) & np.isfinite(q_sim_sub) & np.isfinite(p_sub)
            if not np.any(m):
                return np.nan, np.nan
            rc_obs = runoff_coefficient_p_over_r(p_sub[m], q_obs_sub[m])
            rc_sim = runoff_coefficient_p_over_r(p_sub[m], q_sim_sub[m])
            return float(rc_obs), float(rc_sim)

        print(f"Calibrating basin={basin_id} (window days={n})...")
        best_x, best_f = lsade_optimize(
            eval_trial,
            bounds_low,
            bounds_high,
            max_gens=args.max_gens,
            np_max=args.np_max,
            np_min=args.np_min,
            memory_size=args.memory_size,
            p_best_frac=args.p_best_frac,
            seed=config.seed + 12345 * basin_i,
            progress_label=basin_id,
        )

        best_params = vector_to_params(best_x)
        target_cal_best = -float(best_f)
        target_val_best = float(compute_validation_best(best_x))
        (
            kge_q_cal,
            kge_invq_cal,
            kge_et_cal,
            f_cal,
            lognse_cal,
            kge_bfi_cal,
            bfi_sim_cal,
            bfi_obs_cal,
            kge_logq_cal,
        ) = compute_components_for_indices(
            best_x, train_idx
        )
        (
            kge_q_val,
            kge_invq_val,
            kge_et_val,
            f_val,
            lognse_val,
            kge_bfi_val,
            bfi_sim_val,
            bfi_obs_val,
            kge_logq_val,
        ) = compute_components_for_indices(
            best_x, val_idx
        )
        rc_obs_cal, rc_sim_cal = compute_rc_for_indices(best_x, train_idx)
        rc_obs_val, rc_sim_val = compute_rc_for_indices(best_x, val_idx)

        results.append(
            {
                "basin_id": basin_id,
                # Put low-flow primary metrics first for easier inspection.
                "KGE_invQ_cal": kge_invq_cal,
                "KGE_invQ_val": kge_invq_val,
                "KGE_logQ_cal": kge_logq_cal,
                "KGE_logQ_val": kge_logq_val,
                "alpha1": best_params[0],
                "Sm": best_params[1],
                "alpha2": best_params[2],
                "Kf": best_params[3],
                "Ks": best_params[4],
                "F_cal": f_cal,
                "F_val": f_val,
                "objective_target_cal": target_cal_best,
                "objective_target_val": target_val_best,
                "KGE_Q_cal": kge_q_cal,
                "logNSE_cal": lognse_cal,
                "KGE_BFI_cal": kge_bfi_cal,
                "BFI_sim_cal": bfi_sim_cal,
                "BFI_obs_cal": bfi_obs_cal,
                "RC_obs_cal": rc_obs_cal,
                "RC_sim_cal": rc_sim_cal,
                "KGE_ET_cal": kge_et_cal,
                "KGE_Q_val": kge_q_val,
                "logNSE_val": lognse_val,
                "KGE_BFI_val": kge_bfi_val,
                "BFI_sim_val": bfi_sim_val,
                "BFI_obs_val": bfi_obs_val,
                "RC_obs_val": rc_obs_val,
                "RC_sim_val": rc_sim_val,
                "KGE_ET_val": kge_et_val,
                "train_days": int(train_idx.size),
                "val_days": int(val_idx.size),
                "start_hydro": str(start_hydro),
                "window_start_idx": int(start_idx),
                "window_days": int(n),
                "kge_mode": args.kge_mode,
                "inv_flow_offset": float(args.inv_flow_offset),
                "objective_target": args.objective_target,
                "lognse_offset": float(args.lognse_offset),
                "lh_alpha": float(args.lh_alpha),
                "lh_passes": int(args.lh_passes),
                "sm_mode": args.sm_mode,
                "sm_rel_range": float(args.sm_rel_range),
                "sm_bounds_mode": args.sm_bounds_mode,
                "sm_abs_min": float(args.sm_abs_min),
                "sm_abs_max": float(args.sm_abs_max),
                "q_obs_unit": args.q_obs_unit,
                "area_km2": float(area_km2) if np.isfinite(area_km2) else np.nan,
            }
        )

    out_df = pd.DataFrame(results)
    out_csv_dir = os.path.dirname(args.output_csv)
    if out_csv_dir:
        os.makedirs(out_csv_dir, exist_ok=True)
    out_df.to_csv(args.output_csv, index=False)
    print(f"Wrote results to: {args.output_csv}")
    print(out_df)


# Execute the script entry point
main()
